# GSM 스마트폰 가격 예측 및 가성비 추천 모델링

정리된 팀 경로 `content/`의 GSM 전처리 CSV를 입력으로 사용해 기업용 가격 가이드와 소비자용 예산 추천을 재현한다.

주요 목표:
- 회귀 모델로 `target_price_eur` 예측 및 가격 결정 요인 분석
- 브랜드 중요도와 동일 사양 대비 브랜드 프리미엄 분석
- 클러스터링으로 Entry / Mid-range / Flagship 세그먼트 분류
- `performance_to_price_ratio`, `expected_market_price_eur`, `market_price_discount_ratio` 기반 가성비 이상치 탐지
- 사용자 예산 내 Performance-to-Price Ratio를 최대화하는 최적 모델 추천


In [ ]:
# 1. 라이브러리 임포트 + GitHub 루트 경로 자동 세팅
# 기준 구조: DS_Team7/content, DS_Team7/Preprocessing, DS_Team7/inspection_data, DS_Team7/modeling
from pathlib import Path
import os
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False

REPO_URL = os.environ.get("DS_TEAM7_REPO_URL", "https://github.com/zwonhong/DS_Team7.git")
PROJECT_MARKERS = [
    Path("content/gsm_processed_all(price_prediction).csv"),
    Path("content/gsm_processed(price_prediction).csv"),
    Path("content/gsm_processed_all(recommendation).csv"),
    Path("content/gsm_processed(recommendation).csv"),
    Path("modeling/run_modeling.py"),
]


def has_project_markers(path: Path) -> bool:
    return all((path / marker).exists() for marker in PROJECT_MARKERS)


def safe_resolve(path: Path) -> Path | None:
    try:
        return path.expanduser().resolve()
    except Exception:
        return None


def candidate_roots() -> list[Path]:
    """Return likely DS_Team7 repository roots without using ZIP files."""
    cwd = Path.cwd().resolve()
    env_root = os.environ.get("DS_TEAM7_ROOT")
    bases: list[Path] = []
    if env_root:
        bases.append(Path(env_root))
    bases.extend([
        cwd,
        cwd / "DS_Team7",
        cwd / "DS_Team7-main",
        cwd / "DS_git" / "DS_Team7",
        Path("/content/DS_Team7"),
        Path("/content/DS_Team7-main"),
    ])
    for parent in cwd.parents:
        if str(parent) in {"/", "/Users", "/home", "/private", "/var"}:
            break
        bases.extend([
            parent,
            parent / "DS_Team7",
            parent / "DS_Team7-main",
            parent / "DS_git" / "DS_Team7",
        ])
    seen = set()
    unique = []
    for base in bases:
        resolved = safe_resolve(base)
        if resolved is not None and resolved not in seen:
            seen.add(resolved)
            unique.append(resolved)
    return unique


def find_project_root() -> Path | None:
    for candidate in candidate_roots():
        if has_project_markers(candidate):
            return candidate
    return None


def clone_repo_for_colab() -> Path | None:
    """Clone the GitHub repository in Colab when only the notebook was opened."""
    content_root = Path("/content")
    if not content_root.exists():
        return None
    target = content_root / "DS_Team7"
    if not target.exists():
        print(f"GitHub repository clone: {REPO_URL} -> {target}")
        result = subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(target)],
            text=True,
            capture_output=True,
        )
        if result.returncode != 0:
            print(result.stdout)
            print(result.stderr)
            return None
    else:
        print(f"기존 Colab repository 후보 사용: {target}")
    return target if has_project_markers(target) else None


ROOT_DIR = find_project_root()
if ROOT_DIR is None:
    ROOT_DIR = clone_repo_for_colab()

if ROOT_DIR is None:
    searched = "\n".join(f"- {p}" for p in candidate_roots())
    raise FileNotFoundError(
        "DS_Team7 GitHub 루트 구조를 찾지 못했습니다.\n\n"
        "필요한 구조는 GitHub 화면과 같은 형태입니다:\n"
        "DS_Team7/content/*.csv\n"
        "DS_Team7/Preprocessing/\n"
        "DS_Team7/inspection_data/\n"
        "DS_Team7/modeling/run_modeling.py\n\n"
        "ZIP 파일은 사용하지 않도록 설정했습니다.\n"
        "Colab에서 노트북만 열었다면 첫 셀이 GitHub repo clone을 시도합니다. "
        "clone 후에도 실패한다면 GitHub main branch에 modeling/ 폴더와 content/ CSV가 함께 올라가 있는지 확인하세요.\n\n"
        f"탐색 기준 위치:\n{searched}"
    )

MODELING_DIR = ROOT_DIR / "modeling"
OUTPUT_DIR = MODELING_DIR / "outputs"
PLOT_DIR = OUTPUT_DIR / "plots"
CONTENT_DIR = ROOT_DIR / "content"
PREPROCESSING_DIR = ROOT_DIR / "Preprocessing"
INSPECTION_DIR = ROOT_DIR / "inspection_data"

# 이후 직접 실행하는 셀/명령도 같은 기준 경로를 쓰도록 프로젝트 루트로 이동한다.
os.chdir(ROOT_DIR)

print("ROOT_DIR:", ROOT_DIR)
print("CONTENT_DIR:", CONTENT_DIR, CONTENT_DIR.exists())
print("MODELING_DIR:", MODELING_DIR, MODELING_DIR.exists())
print("PREPROCESSING_DIR:", PREPROCESSING_DIR, PREPROCESSING_DIR.exists())
print("INSPECTION_DIR:", INSPECTION_DIR, INSPECTION_DIR.exists())
print("required content files exist:", all((CONTENT_DIR / name).exists() for name in [
    "gsm_processed_all(price_prediction).csv",
    "gsm_processed(price_prediction).csv",
    "gsm_processed_all(recommendation).csv",
    "gsm_processed(recommendation).csv",
]))


In [ ]:
# 2. 데이터 로드 확인
# 팀원이 정리한 content/ 내부 4개 전처리 CSV의 shape, 결측치, object 컬럼 수를 확인한다.
required_members = [
    "gsm_processed_all(price_prediction).csv",
    "gsm_processed(price_prediction).csv",
    "gsm_processed_all(recommendation).csv",
    "gsm_processed(recommendation).csv",
]

if not CONTENT_DIR.exists():
    raise FileNotFoundError(f"content 폴더가 없습니다: {CONTENT_DIR}")

missing = [member for member in required_members if not (CONTENT_DIR / member).exists()]
if missing:
    raise FileNotFoundError(
        "필수 전처리 CSV가 없습니다: " + ", ".join(missing) +
        "\nGitHub 루트 구조의 DS_Team7/content/ 폴더에 필수 CSV가 있어야 합니다. Colab에서는 1번 셀이 GitHub repo를 clone했는지 확인하세요."
    )

summary_rows = []
print("content files:", sorted(p.name for p in CONTENT_DIR.glob("*.csv")))
for member in required_members:
    df = pd.read_csv(CONTENT_DIR / member)
    summary_rows.append({
        "file": f"content/{member}",
        "rows": df.shape[0],
        "cols": df.shape[1],
        "null_total": int(df.isna().sum().sum()),
        "object_cols": len(df.select_dtypes(include=["object", "string"]).columns),
    })

display(pd.DataFrame(summary_rows))



## 수업 외 라이브러리/함수 설명

- `lightgbm.LGBMRegressor`: gradient boosting tree 기반 회귀 모델이다. `n_estimators`, `learning_rate`, `num_leaves`, `subsample`, `colsample_bytree`를 사용해 트리 수, 학습률, 복잡도, 샘플링 비율을 조절한다. 출처: https://lightgbm.readthedocs.io/
- `RandomForestRegressor`: 여러 의사결정나무를 bagging으로 학습해 비선형 관계와 상호작용을 포착하는 회귀 모델이다. `n_estimators`, `max_depth`, `min_samples_leaf`를 사용했다. 출처: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html
- `KMeans`: 관측치를 k개 중심점에 할당하는 군집화 알고리즘이다. `n_clusters=3`, `n_init=10`, `random_state=42`를 사용했다. 출처: https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html
- `AgglomerativeClustering`: 계층적 병합 군집화 알고리즘이다. `linkage='ward'`는 군집 내 분산 증가를 최소화한다. 출처: https://scikit-learn.org/stable/modules/generated/sklearn.cluster.AgglomerativeClustering.html
- `IsolationForest`: 무작위 분할 트리에서 고립되기 쉬운 샘플을 이상치로 탐지한다. `contamination=0.05`는 약 5%를 이상치 후보로 본다는 의미다. 출처: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html
- `PCA`: 고차원 군집 결과를 2차원으로 시각화하기 위해 사용한 차원 축소 기법이다. 출처: https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html
- `joblib.dump/load`: 학습된 모델 패키지를 저장하고 다시 불러오기 위해 사용한다. 출처: https://joblib.readthedocs.io/


In [ ]:
# 3. 전체 모델링 파이프라인 실행
# 실제 구현은 modeling/run_modeling.py에 있으며, 이 노트북은 해당 스크립트를 실행해 모든 산출물을 재현한다.
# 실행 내용: Regression -> Feature Importance -> Brand Premium -> Clustering -> Outlier -> Recommendation -> Docs
import subprocess
import sys

result = subprocess.run(
    [sys.executable, str(MODELING_DIR / "run_modeling.py"), "--run"],
    cwd=str(ROOT_DIR),
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("run_modeling.py 실행 실패")


In [ ]:
# 4. Task A 회귀 모델 성능 확인
# MAE/RMSE/sMAPE/MASE/R2를 통해 가격 예측 성능을 평가하고, fold별 CV 결과로 5-fold 실행 여부를 확인한다.
holdout = pd.read_csv(OUTPUT_DIR / "regression_holdout_metrics.csv")
cv = pd.read_csv(OUTPUT_DIR / "regression_cv_metrics.csv")
cv_fold = pd.read_csv(OUTPUT_DIR / "regression_cv_fold_metrics.csv")
print("Holdout metrics")
display(holdout.head(8))
print("5-fold CV average metrics")
display(cv.head(8))
print("5-fold CV fold-level metrics sample")
display(cv_fold.head(10))


In [ ]:
# 5. 가격 결정 요인과 브랜드 프리미엄 확인
# Feature importance와 동일 사양 대비 브랜드 잔차 프리미엄을 확인한다.
importance = pd.read_csv(OUTPUT_DIR / "feature_importance_best_target.csv")
brand_premium = pd.read_csv(OUTPUT_DIR / "adjusted_brand_premium_summary.csv")
print("Top feature importance")
display(importance.sort_values("importance", ascending=False).head(12))
print("Adjusted brand premium")
display(brand_premium.head(12))


In [ ]:
# 6. Task B 클러스터링 세그먼트 확인
# Entry / Mid-range / Flagship 세그먼트가 가격과 스펙 점수 기준으로 해석 가능한지 확인한다.
cluster_summary = pd.read_csv(OUTPUT_DIR / "cluster_segment_summary.csv")
cluster_quality = pd.read_csv(OUTPUT_DIR / "cluster_algorithm_comparison.csv")
display(cluster_summary)
display(cluster_quality)


In [ ]:
# 7. Task C 가성비 이상치와 추천 결과 확인
# 이상치 그룹이 실제로 평균 가격은 낮고, 예상 시장가격 대비 할인되어 있으며, PPR은 높은지 확인한다.
# 추천 결과는 예산 이하 후보 중 Performance-to-Price Ratio 내림차순인지 확인한다.
outlier_validation = pd.read_csv(OUTPUT_DIR / "value_outlier_validation_summary.csv")
outlier_methods = pd.read_csv(OUTPUT_DIR / "value_outlier_method_comparison.csv")
rec_validation = pd.read_csv(OUTPUT_DIR / "recommendation_constraint_validation.csv")
recommendations = pd.read_csv(OUTPUT_DIR / "recommendations_all_scenarios.csv")
display(outlier_methods)
display(outlier_validation)
display(rec_validation)
display(recommendations.head(20))


In [ ]:
# 8. 주요 시각화 파일 확인
# 발표에 사용할 핵심 plot이 생성되었는지 확인한다.
plot_files = sorted(PLOT_DIR.glob("*.png"))
for path in plot_files:
    print(path.name)
if plot_files:
    display(Image(filename=str(PLOT_DIR / "lgbm_feature_importance_top15.png")))


In [ ]:
# 9. 직접 실행 가능한 양방향 솔루션 확인
# 기업용 가격 가이드와 소비자용 추천 demo가 정상 출력되는지 확인한다.
result = subprocess.run(
    [sys.executable, str(MODELING_DIR / "two_way_solution.py"), "--mode", "demo"],
    cwd=str(MODELING_DIR),
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("two_way_solution.py demo 실패")


## 결론

- 기업 측면: 회귀 모델과 feature importance를 통해 하드웨어/브랜드 가격 결정 요인과 적정가 기준선을 제공한다.
- 소비자 측면: 클러스터링 세그먼트, 시장 예상가격 대비 저평가 이상치, 예산 제약 PPR 정렬을 통해 최적 대안 모델을 추천한다.
- 과제 조건: scaling/encoding, regression+clustering, 5-fold CV 결과, outputs, detailed comments, 외부 라이브러리 설명을 모두 포함한다.

한계는 EUR 가격 기준, 텍스트 파싱 기반 스펙, 지역/유통 채널 미반영, 구형/신형 모델 혼재다. 최종 GSM 파일에 CPU 벤치마크가 직접 제공되지 않으므로 RAM, storage, camera, battery, network, display 계열 피처와 `spec_score_0_100`을 하드웨어 성능 대리 지표로 사용했다.
